In [ ]:
import numpy as np

encoder_input = np.load('../data/processed/encoder_input.npy')
decoder_input = np.load('../data/processed/decoder_input.npy')
decoder_output = np.load('../data/processed/decoder_output.npy')

In [ ]:
import joblib

idx2word = joblib.load('../data/processed/idx2word.pkl')
word2idx = joblib.load('../data/processed/word2idx.pkl')

In [3]:
from sklearn.model_selection import train_test_split

encoder_input_train, encoder_input_test, decoder_input_train, decoder_input_test, decoder_output_train, decoder_output_test = train_test_split(encoder_input, decoder_input, decoder_output, test_size=0.2, random_state=42)

In [4]:
len(encoder_input_train), len(encoder_input_test), len(decoder_input_train), len(decoder_input_test), len(decoder_output_train), len(decoder_output_test)

(160040, 40010, 160040, 40010, 160040, 40010)

In [5]:
vocab_size = len(word2idx)
vocab_size

26629

In [6]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding

encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(vocab_size, 256)(encoder_inputs)
encoder_lstm = LSTM(256, return_state=True, return_sequences=False)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

In [7]:
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(vocab_size, 256)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [8]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  6,817,024 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  6,817,024 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    525,312 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    525,312 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  6,843,653 │ lstm_1[0][0]      │
│                     │ 26629)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,528,325 (82.12 MB)

 Trainable params: 21,528,325 (82.12 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
decoder_output_train.shape

(160040, 33)

In [10]:
import numpy as np

decoder_output_train_expanded = np.expand_dims(decoder_output_train, -1)
decoder_output_test_expanded = np.expand_dims(decoder_output_test, -1)

In [11]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

In [12]:
history = model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_output_train_expanded,
    validation_data=([encoder_input_test, decoder_input_test], decoder_output_test_expanded),
    batch_size=64,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
)

Epoch 1/15
2501/2501 ━━━━━━━━━━━━━━━━━━━━ 446s 176ms/step - accuracy: 0.7615 - loss: 1.7121 - val_accuracy: 0.7724 - val_loss: 1.5152 - learning_rate: 0.0010
Epoch 2/15
2501/2501 ━━━━━━━━━━━━━━━━━━━━ 445s 178ms/step - accuracy: 0.7758 - loss: 1.4522 - val_accuracy: 0.7772 - val_loss: 1.4307 - learning_rate: 0.0010
Epoch 3/15
2501/2501 ━━━━━━━━━━━━━━━━━━━━ 503s 178ms/step - accuracy: 0.7794 - loss: 1.3742 - val_accuracy: 0.7789 - val_loss: 1.4022 - learning_rate: 0.0010
Epoch 4/15
2501/2501 ━━━━━━━━━━━━━━━━━━━━ 467s 187ms/step - accuracy: 0.7815 - loss: 1.3214 - val_accuracy: 0.7800 - val_loss: 1.3929 - learning_rate: 0.0010
Epoch 5/15
2501/2501 ━━━━━━━━━━━━━━━━━━━━ 445s 178ms/step - accuracy: 0.7833 - loss: 1.2765 - val_accuracy: 0.7803 - val_loss: 1.3934 - learning_rate: 0.0010
Epoch 6/15
2501/2501 ━━━━━━━━━━━━━━━━━━━━ 502s 178ms/step - accuracy: 0.7851 - loss: 1.2351 - val_accuracy: 0.7806 - val_loss: 1.4009 - learning_rate: 0.0010
Epoch 7/15
2501/2501 ━━━━━━━━━━━━━━━━━━━━ 446s 178ms